# Random Forest:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

## Importing Dataset:

In [2]:
data = pd.read_csv("email_spam_features.csv")

## Building Base Model:

In [3]:
X = data.drop(columns=['Category'])
y = data['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [4]:
label = LabelEncoder()

y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [5]:
rf = RandomForestClassifier()

start_time = time.time()
rf.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.79s


In [6]:
y_pred = rf.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 97.87%

Precision: 97.25%

Recall: 84.80%

F1 Score: 90.60%


In [7]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,904,3
Actual spam,19,106


## Hyper-Parameter Tuning:

In [8]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    max_depth = trial.suggest_categorical('max_depth', [3, 4, 5, 6, 7, 8, 9, 10, None])
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
        max_features=max_features,
        class_weight=class_weight
    )

    result = np.mean(cross_val_score(estimator=model, X=X_train, y=y_train, scoring='precision', cv=5))

    return result

study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10, n_jobs=-1)

print("\nBest Parameters:", study.best_params)

[I 2026-10-26 22:30:53,974] A new study created in memory with name: no-name-c487c867-1f59-4519-8c0e-ed75565d584d
[I 2026-10-26 22:31:09,529] Trial 3 finished with value: 0.9629063448403434 and parameters: {'n_estimators': 121, 'criterion': 'log_loss', 'max_depth': 5, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 3 with value: 0.9629063448403434.
[I 2026-10-26 22:31:21,412] Trial 2 finished with value: 0.9673972338519711 and parameters: {'n_estimators': 216, 'criterion': 'log_loss', 'max_depth': None, 'max_features': 'log2', 'class_weight': None}. Best is trial 2 with value: 0.9673972338519711.
[I 2026-10-26 22:31:49,698] Trial 4 finished with value: 0.9654713454875345 and parameters: {'n_estimators': 353, 'criterion': 'log_loss', 'max_depth': 9, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 2 with value: 0.9673972338519711.
[I 2026-10-26 22:31:51,196] Trial 1 finished with value: 0.921594375123787 and parameters: {'n_estimators': 410, 'criterion': 'gini',


Best Parameters: {'n_estimators': 216, 'criterion': 'log_loss', 'max_depth': None, 'max_features': 'log2', 'class_weight': None}


In [9]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    max_depth = trial.suggest_categorical('max_depth', [3, 4, 5, 6, 7, 8, 9, 10, None])
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
        max_features=max_features,
        class_weight=class_weight
    )

    result = np.mean(cross_val_score(estimator=model, X=X_train, y=y_train, scoring='f1', cv=5))

    return result

study = optuna.create_study(direction='maximize', sampler=TPESampler())
study.optimize(objective, n_trials=10, n_jobs=-1)

print("\nBest Parameters:", study.best_params)

[I 2026-10-26 22:33:20,446] A new study created in memory with name: no-name-8265ead7-874b-4089-a9d8-3e84d099aea6
[I 2026-10-26 22:34:08,454] Trial 1 finished with value: 0.888241878572565 and parameters: {'n_estimators': 373, 'criterion': 'gini', 'max_depth': 4, 'max_features': 'log2', 'class_weight': 'balanced'}. Best is trial 1 with value: 0.888241878572565.
[I 2026-10-26 22:34:16,580] Trial 3 finished with value: 0.9255836470886256 and parameters: {'n_estimators': 405, 'criterion': 'entropy', 'max_depth': 7, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 3 with value: 0.9255836470886256.
[I 2026-10-26 22:34:40,801] Trial 5 finished with value: 0.9312452624618999 and parameters: {'n_estimators': 208, 'criterion': 'entropy', 'max_depth': 5, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 5 with value: 0.9312452624618999.
[I 2026-10-26 22:34:53,205] Trial 2 finished with value: 0.9273207512957272 and parameters: {'n_estimators': 885, 'criterion': 'gini


Best Parameters: {'n_estimators': 253, 'criterion': 'gini', 'max_depth': 8, 'max_features': 'log2', 'class_weight': None}


In [10]:
# Priority -> Precision:

Rf = RandomForestClassifier(n_estimators=250, criterion='log_loss', max_depth=None, max_features='log2', class_weight=None)

start_time = time.time()
Rf.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 1.86s


In [11]:
y_pred = Rf.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 98.06%

Precision: 98.17%

Recall: 85.60%

F1 Score: 91.45%


In [12]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,905,2
Actual spam,18,107


In [13]:
# Priority -> F1:

RF = RandomForestClassifier(n_estimators=250, criterion='gini', max_depth=8, max_features='log2', class_weight=None)

start_time = time.time()
RF.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 1.73s


In [14]:
y_pred = RF.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 97.97%

Precision: 97.27%

Recall: 85.60%

F1 Score: 91.06%


In [15]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,904,3
Actual spam,18,107


## Cross-Validation:

In [16]:
# Priority -> Precision:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=RandomForestClassifier(n_estimators=250, criterion='log_loss', max_depth=None, max_features='log2', class_weight=None),
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.43%

Precision: 96.85%

Recall: 90.33%

F1: 93.45%



In [17]:
# Priority -> F1:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=RandomForestClassifier(n_estimators=250, criterion='gini', max_depth=8, max_features='log2', class_weight=None),
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.49%

Precision: 96.86%

Recall: 90.80%

F1: 93.71%

